In [ ]:
from anarcii import Anarcii # https://github.com/oxpig/ANARCII
import pandas as pd
import os 


raw_datasets = pd.read_csv(os.path.join("datasets", "cleaned_seqs_all_seq.csv"), index_col=0)  # Cleaned sequences dataset
seqs = pd.read_csv(os.path.join("datasets", "covid_vaccine_new.sequences.csv"), index_col=0)    # sequences table - imported form ImmuneDB

> Steps required:
1. Pre-process raw sequence.
2. Translate raw sequence.
3. Apply the Anarchii model on the sequence.
4. Compare results with old data.

In [37]:
################################################################
codon_dict = {   'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
                 'TCT': "S", 'TCC': "S", 'TCA': "S", 'TCG': "S",
                 'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',  # * for STOP
                 'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',

                 'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
                 'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
                 'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
                 'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',

                 'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
                 'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
                 'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
                 'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',

                 'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
                 'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
                 'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
                 'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
                }


################################
def use_anarchii(list_seqs: list,
                 model:str = "antibody"):
    
    # Select the type of sequence (antibody, tcr, shark or unknown) and instantiate the model. 
    model = Anarcii(seq_type="antibody")

    # Call the number method on a list of sequences, path to a fasta or PDB file.
    results = model.number(list_seqs)

    return results

######################################
def process_sequence(seq: str) -> str:
    """
    Custom function that process our ImmuneDB sequence for the processing of the Anarcii algorighm.
    (Should work on any NT DNA sequence, it's just won't utilize all of the steps.)
    Steps:
    1. Cheeking for spacers 
    """
    # Counting spcaers ("-") and sequencing unknown AAs ("N")
    n_spacers, n_seqerror = seq.count("-"), seq.count("N")
    div3_spacers, div3_seqerror = seq.count("-") % 3, seq.count("N") % 3

    # If number of spacers isn't dividing by 3 -> raise an error.
    if div3_spacers != 0:
        raise Exception(f"> Number of spacers arent divded by 3 {(n_spacers)}, invalid sequence.")

    seq2translate = seq.replace("-","")
    seq_length = len(seq2translate)
    translated = []
    # Translating sequence
    for i in range(1,seq_length):
        codon = seq2translate[i*3-3:i*3]

        if codon in list(codon_dict.keys()):
            aa = codon_dict[codon]  
        else:
            aa = "N"
         
        translated.append(aa)

    seq_aa = "".join(translated).replace("N","")
    seq_aa_anarchii = use_anarchii(seq_aa)


    return "".join([i[1] for i in seq_aa_anarchii["Sequence"]["numbering"]])

In [ ]:
demo_seq = raw_datasets.sequence[0]
demo_aa = process_sequence(demo_seq)
demo_aa

In [46]:
seqs.columns

Index(['sample_id', 'ai', 'subject_id', 'seq_id', 'partial', 'rev_comp',
       'probable_indel_or_misalign', 'locally_aligned', 'deletions',
       'insertions', 'v_gene', 'j_gene', 'num_gaps', 'seq_start', 'v_match',
       'v_length', 'j_match', 'j_length', 'removed_prefix',
       'removed_prefix_qual', 'v_mutation_fraction', 'pre_cdr3_length',
       'pre_cdr3_match', 'post_cdr3_length', 'post_cdr3_match', 'in_frame',
       'functional', 'stop', 'copy_number', 'cdr3_num_nts', 'cdr3_nt',
       'cdr3_aa', 'sequence', 'quality', 'germline', 'clone_id',
       'mutations_from_clone'],
      dtype='object')

In [45]:
seqs.sequence[0]

'NNNNNNNNNNNNNNNNNNNNNNNNNNN---NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNATTCACCTTC------------AGTAGCTATGCTATGCACTGGGTCCGCCAGGCTCCAGGCAAGGGGCTGGAGTGGGTGGCAGTTATATCATATGAT------GGAAGCAATAAATACTACGCAGACTCCGTGAAG---GGCCGATTCACCATCTCCAGAGACAANTCCAAGAACACGCTGTATCTGCAAATGAACAGCCTGAGAGCTGAGGACACGGCTGTGTATTACTGTGCGAGAGCTAGCCTCGGCAGTGGCTGGTACAGATTTTACTACTANTACNACATNGACGTCTGGGGCAAAGGGACCACGGTCACCGTCTCCTCAN'